In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
file_path = '<CT_RATE_DATASET_DIR>/labels/report_generation/report_generation_test.json'
with open(file_path, 'r') as f:
    data = json.load(f)

In [ ]:
np.random.seed(0)
selected_idx = np.random.choice(np.arange(len(data), step=2), size=1000, replace=False)

In [ ]:
# select all the items with the selected indices where 'id' is 'report_generation_idx
selected_data = [item for item in data if int(item['id'].split('_')[-1]) in selected_idx]

[{'id': 'report_generation_2',
  'image': 'valid_2_a_1.nii.gz',
  'conversations': [{'type': 'report_generation',
    'from': 'human',
    'value': '<image>\nCan you generate the report for the following chest CT image?<report_generation>'},
   {'from': 'gpt',
    'value': 'Findings:  As far as can be seen; A stable soft tissue mass of approximately 5x4x5. On the right, both thyroid glands have increased in size and their parenchyma is heterogeneous. US control is recommended. Trachea and lumen of both main bronchi are open. No occlusive pathology was detected in the trachea and lumen of both main bronchi. Calibration of thoracic main vascular structures is natural. No dilatation was detected in the thoracic aorta. Heart contour size is natural. Pericardial thickening-effusion was not detected. Thoracic esophagus calibration was normal and no significant pathological wall thickening was detected. Sliding type hiatal hernia was observed. No lymph node was detected in mediastinal and bil

In [ ]:
df = pd.read_csv('<CT_RATE_DATASET_DIR>/valid_labels.csv')

In [ ]:
ids = [s['id'] for s in selected_data]
images = [s['image'] for s in selected_data]
query = [s['conversations'][0]['value'] for s in selected_data]
gt = [s['conversations'][1]['value'] for s in selected_data]
df_select = pd.DataFrame(
    {
        'id': ids,
        'VolumeName': images,
        'query': query,
        'gt_report': gt
    }
)

In [167]:
df_full = pd.merge(df, df_select, on='VolumeName', how='inner')

In [ ]:
# Pathology to flip column: for each row select at random on pathology and flip its label, the pathology_to_flip column indicates which pathology has been flipped
PATHOLOGIES_LIST = [
    "Medical material",
    "Arterial wall calcification",
    "Cardiomegaly",
    "Pericardial effusion",
    "Coronary artery wall calcification",
    "Hiatal hernia",
    "Lymphadenopathy",
    "Emphysema",
    "Atelectasis",
    "Lung nodule",
    "Lung opacity",
    "Pulmonary fibrotic sequela",
    "Pleural effusion",
    "Mosaic attenuation pattern",
    "Peribronchial thickening",
    "Consolidation",
    "Bronchiectasis",
    "Interlobular septal thickening",
]
df_full['pathology_to_flip'] = df_full.apply(lambda x: np.random.choice(PATHOLOGIES_LIST), axis=1)

,VolumeName,Medical material,Arterial wall calcification,Cardiomegaly,Pericardial effusion,Coronary artery wall calcification,Hiatal hernia,Lymphadenopathy,Emphysema,Atelectasis,...,Pleural effusion,Mosaic attenuation pattern,Peribronchial thickening,Consolidation,Bronchiectasis,Interlobular septal thickening,id,query,gt_report,pathology_to_flip
0,valid_2_a_1.nii.gz,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,report_generation_2,<image>\nCan you generate the report for the f...,Findings: As far as can be seen; A stable sof...,Arterial wall calcification
1,valid_3_a_1.nii.gz,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,report_generation_4,<image>\nProduce the report for this CT image....,Findings: Trachea was in the midline of both m...,Lung nodule
2,valid_4_b_1.nii.gz,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,report_generation_8,<image>\nPlease generate the report for the ch...,"Findings: Trachea, both main bronchi are open...",Interlobular septal thickening
3,valid_5_a_1.nii.gz,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,report_generation_10,<image>\nProduce the report for this CT scan.<...,Findings: No lymph node was observed in the ax...,Pulmonary fibrotic sequela
4,valid_6_a_1.nii.gz,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,report_generation_12,<image>\nGenerate radiology report for the CT ...,Findings: Trachea and both main bronchi are op...,Medical material
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,valid_1298_a_2.nii.gz,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,report_generation_3026,<image>\nI need a detailed report for the give...,"Findings: Trachea, both main bronchi are open....",Hiatal hernia
996,valid_1301_a_2.nii.gz,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,report_generation_3032,<image>\nProduce the report for this CT volume...,Findings: Trachea and both main bronchi were o...,Cardiomegaly
997,valid_1302_a_2.nii.gz,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,report_generation_3034,<image>\nPlease generate the report for the ch...,Findings: Trachea and main bronchi are open. R...,Pleural effusion
998,valid_1303_a_2.nii.gz,0,1,0,0,1,0,1,1,0,...,0,0,1,0,1,0,report_generation_3036,<image>\nWrite a radiology report for the foll...,Findings: Evaluation of solid organs and major...,Pulmonary fibrotic sequela


In [2]:
import pandas as pd
df_full = pd.read_csv('<CT_RATE_DATASET_DIR>/labels/hallucination_detection_dataset_long.csv')

In [ ]:
# Now generate a "hallu_query"
# For each row get the true value of the pathology to flip, and then generate a hallucinated query by adding "Hint: I think that the patient has [pathology_to_flip]" if the true value is 0, and "Hint: I think that the patient does not have [pathology_to_flip]" if the true value is 1
def generate_hallu_query(row):
    if row[row['pathology_to_flip']] == 0:
        return row['query'] + f" Hint: I think that the patient has {row['pathology_to_flip']}."
    else:
        return row['query'] + f" Hint: I think that the patient does not have any {row['pathology_to_flip']}."
df_full['hallu_query_think'] = df_full.apply(generate_hallu_query, axis=1)
df_full

In [4]:
def generate_correct_hint(row):
    if row[row['pathology_to_flip']] == 1:
        return row['query'] + f" Hint: I think that the patient has {row['pathology_to_flip']}."
    else:
        return row['query'] + f" Hint: I think that the patient does not have any {row['pathology_to_flip']}."
df_full['hallu_query_correct_think'] = df_full.apply(generate_correct_hint, axis=1)

In [9]:
df_full.to_csv('<CT_RATE_DATASET_DIR>/labels/hallucination_detection_dataset_long.csv', index=False)
df_full.to_csv('<RADAGENT_REPO_ROOT>/radagent/evaluation/hallucination_study/hallucination_detection_dataset_long.csv', index=False)

In [58]:
len(list(Path('<RADAGENT_RESULTS_DIR>/ctrate_report_generation/models/2802_v8c_base_f1only_man02_traj02/checkpoints/saved_0149/hallu_corrupted_reports').glob('*/*.json')))

100

In [61]:
with open('<RADAGENT_RESULTS_DIR>/ctrate_report_generation/models/2802_v8c_base_f1only_man02_traj02/checkpoints/saved_0149/hallu_corrupted_reports/trajectory/valid_8_a_1_trajectoryreport_generation_18.json', 'r') as f:
    data = json.load(f)

In [ ]:
content = json.loads(data[2]['content'])
content['action'] == 'call_tool' and content['tool_name'] == 'report_generation_tool'
report = data[3]['content']

'Findings: Trachea and both main bronchi are open. No occlusive pathology was detected in the trachea and both main bronchi. There are emphysematous changes in both lungs. There are atelectasis in the lower lobe of both lungs, the middle lobe of the right lung, and the lingular segment of the left lung upper lobe. There are millimetric nonspecific nodules in both lungs. No mass or infiltrative lesion was detected in both lungs. Mediastinal structures cannot be evaluated optimally because contrast material is not given. As far as can be observed: Heart contour and size are normal. No pleural or pericardial effusion was detected. The widths of the mediastinal main vascular structures are normal. There are atheromatous plaques in the aorta and coronary arteries. There are no pathologically enlarged lymph nodes in the mediastinum and hilar regions. There is a sliding type hiatal hernia at the lower end of the esophagus. No upper abdominal free fluid-collection was detected in the sections.

In [150]:
volume_name = 'valid_380_a_2.nii.gz'
p = str(df_full.loc[df_full['VolumeName'] == volume_name, 'pathology_to_flip'].values[0])
df_full.loc[df_full['VolumeName'] == volume_name , p]

301    0
Name: Pericardial effusion, dtype: int64

In [151]:
df_full.loc[df_full[p]==1,'gt_report'].values[13]

'Findings: The dimensions of the thyroid gland have increased, and a hypodense nodule of 30x40 mm, extending towards the mediastinum, is observed in the left lobe. The cardiothoracic ratio increased in favor of the heart. The diameter of the ascending aorta was 39 mm and increased. Several lymph nodes with a diameter of 6.5 mm are observed in the mediastinum and bilateral hilar regions, the largest of which is in the aortopulmonary window, and no enlarged lymph nodes in pathological size and appearance were detected. Trachea and both main bronchi are open. No occlusive pathology was detected in the trachea and both main bronchi. Pericardial 1 cm thick low-density effusion is observed. Pleural effusion with a thickness of 1.5 cm in the right hemithorax and 1 cm in the left hemithorax is observed. There is minimal effusion in the left major fissure. There is bilateral minimal tubular bronchiectasis and accompanying peribronchial thickness increase. There are increased interlobular septal

In [177]:
manually_corrupted_traces = '<RADAGENT_REPO_ROOT>/radagent/evaluation/hallucination_study/corrupted_traces'
volume_names_corrupted = []
corrupted_reports = []
for p in Path(manually_corrupted_traces).glob('*.json'):
    volume_name = str(p).split('/')[-1]
    volume_name_prefix = volume_name.split('_trajectory')[0]
    with open(p, 'r') as f:
        data = json.load(f)
    content = json.loads(data[2]['content'])
    volume_names_corrupted.append(f'{volume_name_prefix}.nii.gz')
    corrupted_reports.append(data[3]['content'])
    
df_corrupted = pd.DataFrame(
    {
        'VolumeName': volume_names_corrupted,
        'corrupted_report': corrupted_reports
    }
)

df_corrupted_full = pd.merge(df_full, df_corrupted, on='VolumeName', how='inner')
    

In [180]:
df_corrupted_full.to_csv('<CT_RATE_DATASET_DIR>/labels/hallucination_corrupted_reports.csv', index=False)
df_corrupted_full.to_csv('<RADAGENT_REPO_ROOT>/radagent/evaluation/hallucination_study/hallucination_detection_dataset_corrupted_reports.csv', index=False)


In [3]:
import pandas as pd
df = pd.read_csv('<RADAGENT_REPO_ROOT>/radagent/evaluation/hallucination_study/hallucination_detection_dataset_corrupted_reports.csv')

In [ ]:
# copy files from orig_trajectory_files to target_trajectory_files
model_ckpt = '<RADAGENT_RESULTS_DIR>/ctrate_report_generation/models/2802_v8c_base_f1only_man02_traj02/checkpoints/saved_0149/'
orig_inference_folder = 'hallu_orig__20260317_130604'
target_folder = 'hallu_report_orig'
path_to_orig_inference = Path(model_ckpt) / orig_inference_folder
orig_trajectory_files = [str(p) for p in path_to_orig_inference.glob('*/*.json')]
target_trajectory_files = [f.replace(orig_inference_folder, target_folder) for f in orig_trajectory_files]
for orig_file, target_file in zip(orig_trajectory_files, target_trajectory_files):
    volume_name = orig_file.split('/')[-1]
    volume_name_prefix = volume_name.split('_trajectory')[0]
    if f'{volume_name_prefix}.nii.gz' in df['VolumeName'].values:
        target_path = Path(target_file)
        target_path.parent.mkdir(parents=True, exist_ok=True)
        with open(orig_file, 'r') as f:
            data = json.load(f)
        with open(target_file, 'w') as f:
            json.dump(data, f)